# ระดับ 3d2 — Ablation ซ้ำด้วย seed ใหม่ (ตรวจความแปรปรวนจากการฝึก)

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026

**เป้าหมาย:** ระดับ 3d ฝึกทุก variant ด้วย seed เดียว (1234 ตาม `DBBF.yaml`) ผลกลับด้านจาก Table VI แต่ต่างกันไม่เกิน 3 จุด
notebook นี้ฝึก **variant เดิมซ้ำด้วย seed อื่น** เพื่อดูว่า AUC แกว่งตาม seed แค่ไหน และแนวโน้มยังกลับด้านอยู่หรือไม่

โค้ดเหมือน 3d ทุกบรรทัด ต่างกันเฉพาะ `CFG["seed"]` (มีผลกับ head ที่สุ่ม init, ลำดับ batch, Cutout, drop path)
backbone เริ่มจาก FB weight เดียวกันทุก seed

| ค่าเริ่มต้นใน `JOBS` | เวลา (T4 ×2) |
|---|---|
| `fb_ce` × seed 2025, 3407 | ~50 นาที/run |
| `cfb_full` × seed 2025, 3407 | ~100 นาที/run |
| **รวม 4 run** | **~5 ชม.** |

เลือก 2 variant ที่ผลต่างกันมากที่สุดใน 3d (fb_ce สูงสุด, cfb_full ต่ำสุด)
ถ้ามีหลาย Kaggle account แบ่งรันได้ เช่น account A: `JOBS = [("fb_ce", 2025), ("cfb_full", 2025)]` · account B: seed 3407

**ก่อนรัน:** GPU **T4 ×2** · Internet **On** · Add Input = output ของระดับ 3a (เหมือน 3b, 3d)
ผลลัพธ์อยู่ที่ `ablation_seeds/<variant>_s<seed>/best.pth` → ประเมินใน **ระดับ 3f**


## 0. ตั้งค่า

In [ ]:
import os, sys, re, time, json, glob, math, random, shutil, tarfile, hashlib, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR   = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR  = TMP_DIR / "FFDBackbone"
DATA_ROOT = TMP_DIR / "data"                    # จะแตก tar ของระดับ 3a มาไว้ที่นี่
RUN_ROOT  = WORK_DIR / "ablation_seeds"         # ablation_seeds/<variant>_s<seed>/ (ต้องอยู่ใน /kaggle/working)
RUN_DIR   = RUN_ROOT                            # จะถูกตั้งใหม่ต่อ variant ในหัวข้อ 6
OUT_DIR   = WORK_DIR / "results_level3d2"
for d in [TMP_DIR, DATA_ROOT, RUN_ROOT, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1jFIpb4TftJiL82h69c3Roqy8uIzGQP4G"

# --- pre-trained backbone (ผลลัพธ์ของขั้น pre-train ใน paper) ---
# ถ้าอัปเป็น Kaggle Dataset ไว้แล้วให้ใส่ path ตรงนี้ เช่น "/kaggle/input/beit-face/BEiT-1k-Face-55w.tar"
PRETRAIN_PATH = None
PRETRAIN_HINT = "beit-1k-face-55w"              # ชื่อไฟล์ที่ต้องการจาก Google Drive

# --- training (ตาม DBBF.yaml + paper Sec. IV-A2) ---
CFG = dict(
    seed            = 1234,
    image_size      = 224,
    mean            = [0.485, 0.456, 0.406],
    std             = [0.229, 0.224, 0.225],
    num_segments    = 8,
    num_classes     = 2,
    feature_dim     = 768,

    epochs          = 10,                        # paper: 10 สำหรับ backbone ที่ pre-train บนหน้าจริง
    batch_size      = 32,                        # ต่อ GPU → 32 x 2 = 64 ตาม paper
    grad_accum      = 1,
    num_workers     = 4,
    lr              = 5e-5,
    min_lr          = 1e-5,
    warmup_lr       = 1e-6,
    warmup_epochs   = 1,
    weight_decay    = 0.05,
    layer_decay     = 0.75,
    clip_grad       = 1.0,                       # clip_mode: norm ใน yaml
    amp             = True,                      # ต่างจาก repo (fp32) — จำเป็นสำหรับ T4 16 GB
    grad_ckpt       = True,                      # ต่างจาก repo — จำเป็นสำหรับ T4 16 GB
    cutout          = True,                      # cutout: True ใน yaml
    drop_path_rate  = 0.1,

    # EvidenceLoss params (ตรงกับ yaml ทุกตัว)
    edl = dict(num_classes=2, evidence="exp", loss_type="log",
               with_kldiv=False, with_avuloss=True, annealing_method="exp"),

    eval_every      = 1,                         # ประเมิน val ทุกกี่ epoch
    max_train_hours = 10.5,                      # หยุดก่อนชน 12 ชม. ของ Kaggle แล้ว resume รอบหน้า
)

PAPER_FFPP_AUC = 99.36

# --- ablation (Table VI) ---
# dual = 2 branch · dec = L_dec · ufm = uncertainty-based fusion (ไม่มี = บวก feature) · edu = L_EDL+L_EUC (ไม่มี = L_ce)
VARIANT_FLAGS = {
    "fb_ce":       dict(dual=False, dec=False, ufm=False, edu=False),
    "fb_edu":      dict(dual=False, dec=False, ufm=False, edu=True),
    "cfb_base":    dict(dual=True,  dec=False, ufm=False, edu=False),
    "cfb_dec":     dict(dual=True,  dec=True,  ufm=False, edu=False),
    "cfb_dec_ufm": dict(dual=True,  dec=True,  ufm=True,  edu=False),
    "cfb_full":    dict(dual=True,  dec=True,  ufm=True,  edu=True),     # = ระดับ 3b
}
PAPER_TABLE6 = {   # Table VI: CDF, DFDC, FFIW, Avg AUC, Avg ACC
    "fb_ce": (89.18, 84.02, 86.48, 86.56, 79.37), "fb_edu": (89.34, 84.19, 88.76, 87.43, 80.96),
    "cfb_base": (89.77, 83.67, 87.69, 87.04, 79.26), "cfb_dec": (90.31, 84.89, 89.21, 88.14, 79.42),
    "cfb_dec_ufm": (91.16, 84.54, 90.22, 88.64, 80.84), "cfb_full": (90.46, 84.90, 90.97, 88.78, 82.29),
}
# ลำดับความสำคัญ: 2 ตัวแรกคือ baseline หลัก (ไม่มีองค์ประกอบใหม่ของ paper เลย)
# (variant, seed) ที่จะฝึก · seed 1234 ฝึกไปแล้วใน 3b / 3d
JOBS = [("fb_ce", 2025), ("cfb_full", 2025), ("fb_ce", 3407), ("cfb_full", 3407)]
VARIANTS = sorted({v for v, _ in JOBS})
NOTEBOOK_HOURS = 11.3          # เผื่อเวลาให้ commit ก่อนชน 12 ชม. — variant ที่เริ่มไม่ทันจะถูกข้าม (รันต่อ session หน้าได้)
random.seed(CFG["seed"])
assert all(v in VARIANT_FLAGS for v in VARIANTS), "ชื่อ variant ไม่ถูกต้อง"
assert all(sd != 1234 for _, sd in JOBS), "seed 1234 ฝึกไปแล้วใน 3b/3d"
print("ON_KAGGLE:", ON_KAGGLE, "| RUN_ROOT:", RUN_ROOT, "| JOBS:", JOBS)

## 1. ติดตั้ง library

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)

pip_install("gdown")
for mod, pkg in [("timm", "timm"), ("albumentations", "albumentations==1.4.24")]:
    try:
        __import__(mod)
    except ImportError:
        pip_install(pkg)

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import albumentations as alb
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

N_GPU = torch.cuda.device_count()
ENV = {"python": sys.version.split()[0], "torch": torch.__version__, "timm": timm.__version__,
       "albumentations": alb.__version__, "numpy": np.__version__,
       "n_gpu": N_GPU, "gpu": [torch.cuda.get_device_name(i) for i in range(N_GPU)]}
print(ENV)
assert N_GPU >= 1, "ต้องเปิด GPU ใน Kaggle (Settings → Accelerator)"

## 2. Clone repo + patch
ใช้โค้ด BEiT v2 ของ repo ตรง ๆ (`models/lib/BEiT_v2/modeling_finetune.py`) และ `EvidenceLoss`
(`common/losses/edl_loss.py`) เพื่อให้สถาปัตยกรรมและ loss ตรงกับต้นฉบับ 100%

In [ ]:
def find_repo_in_input():
    """หา FFDBackbone ที่ Add Input ไว้ (โหมดออฟไลน์) — ค้นแค่ 2 ชั้นแรก ไม่ไล่ทั้ง dataset"""
    if not INPUT_DIR.exists():
        return None
    marker = Path("models") / "lib" / "BEiT_v2" / "modeling_finetune.py"
    for d in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
        cands = [d] + sorted(p for p in d.iterdir() if p.is_dir())[:50]
        for c in cands:
            if (c / marker).exists():
                return c
    return None

def get_repo():
    """1) ที่ clone ไว้แล้ว  2) จาก /kaggle/input (ออฟไลน์)  3) git clone"""
    if (REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py").exists():
        return "cached"
    src = find_repo_in_input()
    if src is not None:
        print("พบ repo ใน Add Input (โหมดออฟไลน์):", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
        return "input"
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone repo ไม่สำเร็จ:\n  " + (r.stderr or "").strip() + "\n\n"
            "ถ้าขึ้น 'Could not resolve host' แปลว่า Internet ปิดอยู่ — แก้ได้ 2 ทาง\n"
            "  (ก) เปิดเน็ต: แผงขวาของ Kaggle -> Notebook options -> Internet = On\n"
            "      (ต้องยืนยันเบอร์โทรที่ kaggle.com/settings ก่อน ถ้ายังไม่เคยทำ)\n"
            "  (ข) โหมดออฟไลน์: ดาวน์โหลด zip ของ repo จาก\n"
            "      https://github.com/zhenglab/FFDBackbone/archive/refs/heads/main.zip\n"
            "      -> อัปเป็น Kaggle Dataset -> Add Input  (cell นี้จะหาเจอเอง)")
    return "clone"

REPO_SOURCE = get_repo()
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "(จาก Add Input — ไม่มี .git)"

def patch_repo(repo: Path):
    n = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8"); n += 1
    return n

print("repo commit:", REPO_COMMIT, "| patched files:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import importlib.util
def load_module_from_file(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

modeling_finetune = load_module_from_file("beitv2_modeling_finetune",
                                          REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
edl_mod = load_module_from_file("ffd_edl_loss", REPO_DIR / "common" / "losses" / "edl_loss.py")
EvidenceLoss = edl_mod.EvidenceLoss
print("loaded BEiT v2 + EvidenceLoss จาก repo")

## 3. หาและแตกข้อมูลที่เตรียมไว้จากระดับ 3a

รองรับทั้งสองแบบที่ระดับ 3a ผลิตได้:
- **tar** (`original.tar`, `Deepfakes.tar`, …) → แตกลง `/tmp/ffd/data`
- **ไฟล์ย่อย** (`FF++/face_v1/...`) → ใช้ path ของ input ตรง ๆ (read-only ก็ใช้ได้)

ถ้าแบ่งรันระดับ 3a หลายรอบ ให้ Add Input ทุก output — cell นี้รวมให้เอง

In [ ]:
FACE_ROOT = None

tars = sorted(INPUT_DIR.rglob("*.tar")) if INPUT_DIR.exists() else []
tars = [t for t in tars if t.stem in ["meta", "original"] + ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]]
if tars:
    print("พบ tar จากระดับ 3a:", *[f"{t.name} ({t.stat().st_size/1e9:.2f} GB)  {t.parent}" for t in tars], sep="\n  ")
    for t in tars:
        marker = DATA_ROOT / f".extracted_{t.parent.name}_{t.stem}"
        if marker.exists():
            continue
        print("แตก", t.name, "...")
        with tarfile.open(t) as tf:
            tf.extractall(DATA_ROOT)
        marker.touch()
    FACE_ROOT = DATA_ROOT / "FF++" / "face_v1"
else:
    hits = sorted(INPUT_DIR.rglob("face_v1/splits/train.json")) if INPUT_DIR.exists() else []
    if hits:
        FACE_ROOT = hits[0].parent.parent
        print("ใช้ไฟล์ย่อยจาก input:", FACE_ROOT)

if FACE_ROOT is None or not (FACE_ROOT / "splits" / "train.json").exists():
    raise RuntimeError("ไม่พบข้อมูลจากระดับ 3a — Add Input เป็น output ของ notebook FFD_level3a แล้วรันใหม่")

splits = {s: json.load(open(FACE_ROOT / "splits" / f"{s}.json")) for s in ["train", "val", "test"]}
print("\nFACE_ROOT:", FACE_ROOT)
print("split sizes:", {s: len(v) for s, v in splits.items()})
meta_path = FACE_ROOT / "prepare_meta.json"
if meta_path.exists():
    print("prepare_meta:", json.dumps(json.load(open(meta_path)), ensure_ascii=False)[:400])

### 3.1 สร้างรายการวิดีโอตาม official split + ตรวจความครบถ้วน
ทำตาม `dataloader_util.get_FF_list()` ทุกบรรทัด: real = `{target}`, `{source}` ของทุกคู่ ·
fake = `{target}_{source}`, `{source}_{target}` ของทุก method

In [ ]:
COMPRESSION = "c23"
FF_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
REAL_LABEL, FAKE_LABEL = 0, 1
REAL_REL = f"original_sequences/youtube/{COMPRESSION}/videos"

def get_FF_list(split):
    real_names, fake_names = [], []
    for t, s in splits[split]:
        real_names += [t, s]
        fake_names += [f"{t}_{s}", f"{s}_{t}"]
    rows = [{"rel": f"{REAL_REL}/{x}", "label": REAL_LABEL, "method": "original", "video": x} for x in real_names]
    for m in FF_METHODS:
        rows += [{"rel": f"manipulated_sequences/{m}/{COMPRESSION}/videos/{x}",
                  "label": FAKE_LABEL, "method": m, "video": x} for x in fake_names]
    return rows

def frames_of(rel, k):
    """เหมือน get_sampled_idx(): step = N // k แล้วเอา k ตัวแรก"""
    d = FACE_ROOT / rel / "frame"
    if not d.is_dir():
        return []
    idxs = sorted(int(p.stem) for p in d.glob("*.png"))
    if not idxs:
        return []
    if len(idxs) >= k:
        step = len(idxs) // k
        sel = idxs[::step][:k]
    else:                               # วนซ้ำ frame เดิมให้ครบ k (check_frame_len ของ repo)
        sel = sorted(np.resize(np.array(idxs), k).tolist())
    return [str(d / f"{i}.png") for i in sorted(sel)]

K_FRAMES = CFG["num_segments"]
index = {}
for sp in ["train", "val", "test"]:
    rows = []
    for r in get_FF_list(sp):
        fr = frames_of(r["rel"], K_FRAMES)
        rows.append({**r, "n_frames": len(fr), "frames": fr})
    index[sp] = pd.DataFrame(rows)
    df = index[sp]
    ok = df["n_frames"] == K_FRAMES
    print(f"{sp:<5} วิดีโอ {len(df):>5}  ใช้ได้ {ok.sum():>5}  ขาด {(~ok).sum():>4}  "
          f"real {int((df['label']==0).sum()):>4} fake {int((df['label']==1).sum()):>5}")
    index[sp] = df[ok].reset_index(drop=True)

expected = {"train": 3600, "val": 700, "test": 700}
for sp, n in expected.items():
    got = len(index[sp])
    flag = "✓" if got == n else f"✗ (คาด {n})"
    print(f"  {sp}: {got} วิดีโอ {flag}")

# train = frame-level (เหมือน Image_dataset ของ repo), val/test = video-level
train_frames = [(f, int(l)) for fs, l in zip(index["train"]["frames"], index["train"]["label"]) for f in fs]
print(f"\nframe สำหรับเทรน: {len(train_frames):,}  (real {sum(1 for _,l in train_frames if l==0):,} / fake {sum(1 for _,l in train_frames if l==1):,})")

## 4. Backbone ที่ pre-train บนใบหน้าจริง (`BEiT-1k-Face-55w.tar`)

นี่คือ **ผลลัพธ์ของขั้น pre-train** ใน paper: ViT-B/16 ที่ผ่าน BEiT v2 MIM บนภาพใบหน้าจริง 550k ภาพ
(CelebA + CelebV-Text + FFHQ) บน 4×A100 — ผู้เขียนไม่ได้แนบสคริปต์ pre-train มาใน repo จึงใช้ weight สำเร็จรูป

⚠️ `DDBF_BEiT_v2` ของ repo โหลดแบบ `strict=False` และไม่เช็กผล — ถ้าไฟล์ผิด backbone จะเป็น **weight สุ่ม**
โดยไม่มี error ใด ๆ cell นี้จึงพิมพ์ missing/unexpected keys ออกมาให้ตรวจ

In [ ]:
CKPT_EXTS = (".tar", ".pth", ".pt", ".ckpt", ".bin")

def find_or_download_pretrain():
    if PRETRAIN_PATH and Path(PRETRAIN_PATH).exists():
        return Path(PRETRAIN_PATH)
    local = [p for root in [str(INPUT_DIR), str(TMP_DIR / "weights")] if Path(root).exists()
             for p in Path(root).rglob("*") if p.is_file()
             and p.name.lower().endswith(CKPT_EXTS) and PRETRAIN_HINT in p.name.lower()]
    if local:
        print("พบ pre-trained backbone ในเครื่อง:", *local, sep="\n  ")
        return local[0]
    try:
        import gdown, inspect
    except ImportError:
        raise RuntimeError("ไม่มี gdown และติดตั้งไม่ได้ (เน็ตปิด) — ดาวน์โหลด BEiT-1k-Face-55w.tar เอง "
                           "แล้วอัปเป็น Kaggle Dataset -> Add Input")
    (TMP_DIR / "weights").mkdir(parents=True, exist_ok=True)
    kw = dict(url=GDRIVE_FOLDER, skip_download=True, quiet=True)
    if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
        kw["remaining_ok"] = True
    files = gdown.download_folder(**kw) or []
    entries = [(getattr(f, "id", None) or f[0], str(getattr(f, "path", None) or f[1])) for f in files]
    print("ไฟล์ใน Drive:", *[p for _, p in entries], sep="\n  ")
    cands = [(i, p) for i, p in entries if PRETRAIN_HINT in p.lower()]
    if not cands:
        raise RuntimeError(f"ไม่พบไฟล์ที่มี '{PRETRAIN_HINT}' ใน Drive — ดาวน์โหลดเอง อัปเป็น Kaggle Dataset "
                           "แล้วตั้ง PRETRAIN_PATH")
    fid, rel = cands[0]
    out = TMP_DIR / "weights" / Path(rel).name
    print("ดาวน์โหลด:", rel)
    gdown.download(id=fid, output=str(out), quiet=False)
    if not out.exists() or out.stat().st_size < 10_000_000:
        raise RuntimeError("ดาวน์โหลดไม่สำเร็จ (อาจติดโควตา Google Drive) — ดาวน์โหลดเองแล้วตั้ง PRETRAIN_PATH")
    return out

pretrain_file = find_or_download_pretrain()
h = hashlib.sha256()
with open(pretrain_file, "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""):
        h.update(b)
PRETRAIN_SHA = h.hexdigest()
print(f"\npre-trained backbone: {pretrain_file} ({pretrain_file.stat().st_size/1e6:.0f} MB)  sha256: {PRETRAIN_SHA[:16]}")

# --- ตรวจว่าโหลดเข้า BEiT ได้จริง ---
BEiT_Config = {"drop_path_rate": CFG["drop_path_rate"], "use_mean_pooling": False, "init_values": 0.1,
               "qkv_bias": True, "use_abs_pos_emb": False, "use_rel_pos_bias": True,
               "use_shared_rel_pos_bias": False}
_probe = modeling_finetune.beit_base_patch16_224(**BEiT_Config)
_probe.head = nn.Identity()
_ck = torch.load(pretrain_file, map_location="cpu", weights_only=False)
_sd = _ck["model"] if isinstance(_ck, dict) and "model" in _ck else _ck
_msg = _probe.load_state_dict(_sd, strict=False)
_body = [k for k in _probe.state_dict() if not k.startswith(("head", "fc_norm"))]
_loaded = [k for k in _body if k not in set(_msg.missing_keys)]
print(f"โหลด weight: {len(_loaded)}/{len(_body)} keys ของ backbone")
print("missing  :", _msg.missing_keys[:8], "..." if len(_msg.missing_keys) > 8 else "")
print("unexpected:", _msg.unexpected_keys[:8], "..." if len(_msg.unexpected_keys) > 8 else "")
assert len(_loaded) / len(_body) > 0.9, "weight เข้าไม่ถึง 90% — น่าจะเลือกไฟล์ผิด"
del _probe, _ck, _sd

## 5. เขียนสคริปต์เทรน (DDP) — เหมือนระดับ 3b ทุกบรรทัด ยกเว้นส่วนที่เปิด/ปิดได้

สิ่งที่เปลี่ยนจาก 3b มีแค่ 2 จุด (ควบคุมด้วย `flags` ใน config):

1. **โมเดล** — `dual=False` ใช้ backbone เดียว (= FB) · `ufm=False` รวม feature แบบ `f_M + f_A`
2. **loss** — `edu=False` ใช้ `L_ce` แทน `L_EDU` ใน branch · `dec=False` ไม่บวก `L_dec` (ยังคำนวณ Pearson ไว้ดูอย่างเดียว)

การเลือก checkpoint และ val/test AUC ใช้ output ตัวเดียวกับ 3b (fused head สำหรับ 2 branch, main สำหรับ 1 branch)
ส่วน **main branch** ตาม paper จะประเมินใหม่ทุกตัวในระดับ 3e


In [ ]:
SCRIPT = RUN_ROOT / "train_ablation.py"
SCRIPT.write_text(r"""
import os, sys, json, math, time, random, argparse
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.distributed as dist
import cv2, albumentations as alb
from albumentations.pytorch.transforms import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from PIL import Image
from sklearn.metrics import roc_auc_score
import importlib.util

ap = argparse.ArgumentParser(); ap.add_argument('--cfg', required=True)
A = json.load(open(ap.parse_args().cfg))
CFG, REPO_DIR = A['cfg'], Path(A['repo_dir'])
RUN_DIR = Path(A['run_dir']); RUN_DIR.mkdir(parents=True, exist_ok=True)
FL = A.get('flags', dict(dual=True, dec=True, ufm=True, edu=True))
VARIANT = A.get('variant', 'cfb_full')
sys.path.insert(0, str(REPO_DIR))

def load_mod(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

MF  = load_mod('beitv2_mf',  REPO_DIR/'models'/'lib'/'BEiT_v2'/'modeling_finetune.py')
EDL = load_mod('ffd_edl',    REPO_DIR/'common'/'losses'/'edl_loss.py')

LOCAL_RANK = int(os.environ.get('LOCAL_RANK', 0))
WORLD      = int(os.environ.get('WORLD_SIZE', 1))
DDP        = WORLD > 1
if DDP:
    dist.init_process_group(backend='nccl', init_method='env://')
torch.cuda.set_device(LOCAL_RANK)
DEV = torch.device('cuda', LOCAL_RANK)
IS_MAIN = LOCAL_RANK == 0

seed = CFG['seed'] + LOCAL_RANK
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def log(*a):
    if IS_MAIN:
        msg = ' '.join(str(x) for x in a)
        print(msg, flush=True)
        with open(RUN_DIR/'train.log', 'a') as f: f.write(msg + '\n')

# ---------------- data ----------------
def _jpeg(lo, hi, p):
    # albumentations >=1.4.21 เปลี่ยนชื่อ arg เป็น quality_range
    try:
        return alb.ImageCompression(quality_range=(lo, hi), p=p)
    except TypeError:
        return alb.ImageCompression(quality_lower=lo, quality_upper=hi, p=p)

def tf_train(cfg):
    # ตรงกับ create_base_transforms(split='train') ของ repo ทุกตัว
    return alb.Compose([
        alb.RGBShift((-20,20),(-20,20),(-20,20), p=0.3),
        alb.HueSaturationValue(hue_shift_limit=(-0.3,0.3), sat_shift_limit=(-0.3,0.3),
                               val_shift_limit=(-0.3,0.3), p=0.3),
        alb.RandomBrightnessContrast(brightness_limit=(-0.3,0.3), contrast_limit=(-0.3,0.3), p=0.3),
        _jpeg(40, 100, 0.5),
        alb.HorizontalFlip(),
        alb.ToGray(p=0.01),
        alb.Resize(cfg['image_size'], cfg['image_size']),
        alb.Normalize(mean=cfg['mean'], std=cfg['std']),
        ToTensorV2(),
    ])

def tf_eval(cfg):
    return alb.Compose([alb.Resize(cfg['image_size'], cfg['image_size']),
                        alb.Normalize(mean=cfg['mean'], std=cfg['std']), ToTensorV2()])

class Cutout:
    # map_util.Cutout ของ repo (คัดลอกตรง ๆ): 1-3 หลุม ขนาดสุ่ม 2%-60% ของด้าน
    def __call__(self, imgs):
        h, w = imgs[0].size(1), imgs[0].size(2)
        n_holes = np.random.randint(1, 4)
        length = int(h * np.random.randint(1, 30) * 0.01) * 2
        mask = np.ones((h, w), np.float32)
        for _ in range(n_holes):
            y, x = np.random.randint(h), np.random.randint(w)
            y1, y2 = np.clip(y-length//2, 0, h), np.clip(y+length//2, 0, h)
            x1, x2 = np.clip(x-length//2, 0, w), np.clip(x+length//2, 0, w)
            mask[y1:y2, x1:x2] = 0.
        return imgs * torch.from_numpy(mask).expand_as(imgs)

class FrameDS(Dataset):
    # train: 1 frame = 1 sample (เหมือน Image_dataset ของ repo) -> [1,3,H,W]
    def __init__(self, items, cfg):
        self.items, self.tf = items, tf_train(cfg)
        self.cut = Cutout() if cfg['cutout'] else None
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        img = np.asarray(Image.open(p).convert('RGB'))
        x = self.tf(image=img)['image'].unsqueeze(0)       # [1,3,H,W]
        if self.cut is not None: x = self.cut(x)
        return x, y

class VideoDS(Dataset):
    # val/test: 8 frame = 1 sample -> [8,3,H,W]
    def __init__(self, videos, cfg):
        self.videos, self.tf, self.k = videos, tf_eval(cfg), cfg['num_segments']
    def __len__(self): return len(self.videos)
    def __getitem__(self, i):
        fs, y = self.videos[i]['frames'], self.videos[i]['label']
        xs = [self.tf(image=np.asarray(Image.open(p).convert('RGB')))['image'] for p in fs]
        return torch.stack(xs), y, i

# ---------------- model (= DDBF_BEiT_v2 ของ repo) ----------------
BEiT_Config = {'drop_path_rate': CFG['drop_path_rate'], 'use_mean_pooling': False, 'init_values': 0.1,
               'qkv_bias': True, 'use_abs_pos_emb': False, 'use_rel_pos_bias': True,
               'use_shared_rel_pos_bias': False}

def enable_grad_ckpt(vit):
    # BEiT ของ repo ไม่มี gradient checkpointing -> patch forward_features ให้ใช้
    from torch.utils.checkpoint import checkpoint
    def ff(self, x, return_patch_tokens=False, return_all_tokens=False, **kw):
        x = self.patch_embed(x)
        B = x.size(0)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)
        if self.pos_embed is not None: x = x + self.pos_embed
        x = self.pos_drop(x)
        rpb = self.rel_pos_bias() if self.rel_pos_bias is not None else None
        for blk in self.blocks:
            if self.training:
                x = checkpoint(lambda t, b=blk: b(t, rel_pos_bias=rpb), x, use_reentrant=False)
            else:
                x = blk(x, rel_pos_bias=rpb)
        x = self.norm(x)
        return x[:, 0]
    vit.forward_features = ff.__get__(vit, type(vit))
    return vit

def pearson_correlation_loss(x, y):          # models/custom.py ของ repo
    xm = x.sub(x.mean(-1, keepdim=True)); ym = y.sub(y.mean(-1, keepdim=True))
    cov = torch.bmm(xm.unsqueeze(1), ym.unsqueeze(2)).squeeze() / x.size(-1)
    r = (cov / (x.std(-1) * y.std(-1))).mean()
    return torch.clamp(r, -1.0, 1.0)

def uncertainty(out):                        # Eq. 3-4 (K=2)
    alpha = torch.exp(torch.clamp(out, -10, 10)) + 1
    return 2 / torch.sum(alpha, dim=1, keepdim=True)

class DDBF_BEiT_v2(nn.Module):
    # = DDBF_BEiT_v2 ของ repo เมื่อ flags ครบทุกตัว · ปิดบางส่วนได้เพื่อทำ ablation (Table VI)
    def __init__(self, cfg, pretrained_path=None, grad_ckpt=False, flags=None):
        super().__init__()
        self.fl = dict(flags or FL)
        self.backbone_model = MF.beit_base_patch16_224(**BEiT_Config)
        self.backbone_model.head = nn.Identity()
        if self.fl['dual']:
            self.backbone_model2 = MF.beit_base_patch16_224(**BEiT_Config)
            self.backbone_model2.head = nn.Identity()
        if pretrained_path:
            ck = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
            m1 = self.backbone_model.load_state_dict(sd, strict=False)
            if self.fl['dual']:
                self.backbone_model2.load_state_dict(sd, strict=False)
            log('load pretrain | missing:', len(m1.missing_keys), 'unexpected:', len(m1.unexpected_keys))
        if grad_ckpt:
            enable_grad_ckpt(self.backbone_model)
            if self.fl['dual']: enable_grad_ckpt(self.backbone_model2)
        self.norm_for_cls = nn.LayerNorm(cfg['feature_dim'])
        self.header = nn.Sequential(nn.Linear(cfg['feature_dim'], cfg['num_classes']))
    def forward(self, x):
        B, T = x.size(0), x.size(1)
        x = x.flatten(0, 1)
        f1 = self.backbone_model(x)
        p1 = self.header(self.norm_for_cls(f1))
        if not self.fl['dual']:                                   # FB: backbone เดียว + FC
            return p1.view(B, T, -1), p1.view(B, T, -1), None, f1.new_zeros(())
        f2 = self.backbone_model2(x)
        ploss = pearson_correlation_loss(f1.detach().float(), f2.float())   # gradient-stop ที่ main
        p2 = self.header(self.norm_for_cls(f2))
        if self.fl['ufm']:                                        # UFM (Eq. 3-5)
            u = torch.cat([uncertainty(p1.float()), uncertainty(p2.float())], dim=-1)
            w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
            fused = torch.matmul(w.unsqueeze(1), torch.stack((f1, f2), dim=1)).squeeze(1)
        else:                                                     # ไม่มี UFM → บวก feature (caption Table VI)
            fused = f1 + f2
        p = self.header(self.norm_for_cls(fused))
        return p.view(B, T, -1), p1.view(B, T, -1), p2.view(B, T, -1), ploss

# ---------------- optimizer: layer-wise lr decay ----------------
NO_DECAY = ('bias', 'norm', 'cls_token', 'pos_embed', 'relative_position_bias_table', 'gamma_')

def layer_id_of(name, n_layers=12):
    for pre in ('backbone_model2.', 'backbone_model.'):
        if name.startswith(pre): name = name[len(pre):]; break
    if name.startswith(('cls_token', 'pos_embed', 'patch_embed', 'rel_pos_bias')): return 0
    if name.startswith('blocks.'): return int(name.split('.')[1]) + 1
    return n_layers + 1

def build_param_groups(model, cfg, n_layers=12):
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        lid = layer_id_of(n, n_layers)
        wd = 0.0 if (p.ndim == 1 or any(k in n for k in NO_DECAY)) else cfg['weight_decay']
        key = (lid, wd)
        if key not in groups:
            groups[key] = {'params': [], 'weight_decay': wd,
                           'lr_scale': cfg['layer_decay'] ** (n_layers + 1 - lid)}
        groups[key]['params'].append(p)
    return list(groups.values())

def lr_at(step, total_steps, warm_steps, cfg):
    if step < warm_steps:
        return cfg['warmup_lr'] + (cfg['lr'] - cfg['warmup_lr']) * step / max(1, warm_steps)
    t = (step - warm_steps) / max(1, total_steps - warm_steps)
    return cfg['min_lr'] + (cfg['lr'] - cfg['min_lr']) * 0.5 * (1 + math.cos(math.pi * t))

# ---------------- eval ----------------
@torch.no_grad()
def evaluate(model, loader, n_videos):
    model.eval()
    scores = torch.zeros(n_videos, device=DEV); labels = torch.zeros(n_videos, device=DEV)
    seen   = torch.zeros(n_videos, device=DEV)
    for x, y, idx in loader:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=CFG['amp']):
            out, _, _, _ = model(x)
        p = torch.softmax(out.float(), dim=-1)[..., 1].mean(dim=1)     # เฉลี่ย 8 frame = score วิดีโอ
        idx = idx.to(DEV)
        scores[idx] = p; labels[idx] = y.to(DEV).float(); seen[idx] = 1
    if DDP:
        for t in (scores, labels, seen): dist.all_reduce(t, op=dist.ReduceOp.SUM)
    m = seen > 0
    # DistributedSampler เติม sample ซ้ำให้หารลงตัว -> index ที่ซ้ำถูกบวก 2 ครั้ง จึงหารด้วย seen
    scores = torch.where(m, scores / seen.clamp(min=1), scores)
    labels = torch.where(m, labels / seen.clamp(min=1), labels)
    s, l = scores[m].cpu().numpy(), labels[m].cpu().numpy().round().astype(int)
    auc = roc_auc_score(l, s) if len(set(l.tolist())) > 1 else float('nan')
    acc = float(((s >= 0.5).astype(int) == l).mean())
    return auc, acc, s, l

# ---------------- main ----------------
def main():
    train_items = [tuple(x) for x in A['train_items']]
    val_videos, test_videos = A['val_videos'], A['test_videos']

    tr_ds = FrameDS(train_items, CFG)
    tr_sampler = DistributedSampler(tr_ds, shuffle=True, drop_last=True) if DDP else None
    tr_ld = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=tr_sampler is None,
                       sampler=tr_sampler, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    def vloader(vids):
        ds = VideoDS(vids, CFG)
        sp = DistributedSampler(ds, shuffle=False) if DDP else None
        return DataLoader(ds, batch_size=max(1, CFG['batch_size'] // CFG['num_segments']),
                          shuffle=False, sampler=sp, num_workers=CFG['num_workers'], pin_memory=True)
    va_ld, te_ld = vloader(val_videos), vloader(test_videos)

    model = DDBF_BEiT_v2(CFG, A['pretrain_path'], CFG['grad_ckpt'], FL).to(DEV)
    log(f'variant {VARIANT} | flags {FL}')
    if DDP:
        model = nn.parallel.DistributedDataParallel(model, device_ids=[LOCAL_RANK],
                                                    find_unused_parameters=False)
    core = model.module if DDP else model

    import timm.optim as topt
    pg = build_param_groups(core, CFG)
    for g in pg: g['lr'] = CFG['lr'] * g['lr_scale']
    optimizer = topt.Lamb(pg, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=CFG['amp'])
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=CFG['amp'])

    crit_ce  = nn.CrossEntropyLoss().to(DEV)
    crit_edl = EDL.EvidenceLoss(**CFG['edl']).to(DEV)

    steps_per_epoch = len(tr_ld) // CFG['grad_accum']
    total_steps = steps_per_epoch * CFG['epochs']
    warm_steps  = steps_per_epoch * CFG['warmup_epochs']

    start_epoch, best_auc, history = 1, 0.0, []
    last = RUN_DIR / 'last.pth'
    if last.exists():
        ck = torch.load(last, map_location='cpu', weights_only=False)
        core.load_state_dict(ck['state_dict'])
        optimizer.load_state_dict(ck['optimizer'])
        if ck.get('scaler'): scaler.load_state_dict(ck['scaler'])
        start_epoch, best_auc, history = ck['epoch'] + 1, ck.get('best_auc', 0.0), ck.get('history', [])
        log(f'resume จาก epoch {ck["epoch"]} (best val AUC {100*best_auc:.2f})')

    def sum_edl(out, y, ep):
        d = crit_edl(out, y, epoch=ep, total_epoch=CFG['epochs'])
        return sum(v.mean() for k, v in d.items() if 'loss' in k and torch.is_tensor(v))

    t_start = time.time()
    for epoch in range(start_epoch, CFG['epochs'] + 1):
        if DDP: tr_sampler.set_epoch(epoch)
        model.train()
        agg = {k: 0.0 for k in ['loss', 'ce', 'edl1', 'edl2', 'pear', 'acc']}
        n = 0
        optimizer.zero_grad(set_to_none=True)
        t0 = time.time()
        for it, (x, y) in enumerate(tr_ld):
            gstep = (epoch - 1) * steps_per_epoch + it // CFG['grad_accum']
            lr_now = lr_at(gstep, total_steps, warm_steps, CFG)
            for g in optimizer.param_groups: g['lr'] = lr_now * g['lr_scale']

            x = x.to(DEV, non_blocking=True); y = y.to(DEV, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16, enabled=CFG['amp']):
                out, out_m, out_a, ploss = model(x)
            T = out.size(1)
            yr = y.unsqueeze(1).repeat(1, T).flatten(0, 1)
            o_f, o_m = out.flatten(0,1).float(), out_m.flatten(0,1).float()
            zero = o_f.new_zeros(())
            if FL['dual']:
                o_a = out_a.flatten(0,1).float()
                branch = (lambda o: sum_edl(o, yr, epoch)) if FL['edu'] else (lambda o: crit_ce(o, yr))
                l_ce   = crit_ce(o_f, yr)                        # L^F_ce (fused head)
                l_edl1 = branch(o_m)                             # L^M_EDU หรือ L^M_ce
                l_edl2 = branch(o_a)                             # L^A_EDU หรือ L^A_ce
                loss   = l_ce + l_edl1 + l_edl2 + (ploss if FL['dec'] else zero)   # Eq. 9 เมื่อครบทุกตัว
            else:                                                # FB: loss เดียวที่ head เดียว
                l_ce   = zero if FL['edu'] else crit_ce(o_m, yr)
                l_edl1 = sum_edl(o_m, yr, epoch) if FL['edu'] else zero
                l_edl2 = zero
                loss   = l_edl1 if FL['edu'] else l_ce
            scaler.scale(loss / CFG['grad_accum']).backward()
            if (it + 1) % CFG['grad_accum'] == 0:
                if CFG['clip_grad']:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['clip_grad'])
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)

            bs = y.size(0); n += bs
            agg['loss'] += loss.item()*bs; agg['ce'] += l_ce.item()*bs
            agg['edl1'] += l_edl1.item()*bs; agg['edl2'] += l_edl2.item()*bs
            agg['pear'] += ploss.item()*bs
            agg['acc'] += (o_f.argmax(1) == yr).float().mean().item()*bs
            if IS_MAIN and it % 50 == 0:
                log(f'E{epoch} [{it}/{len(tr_ld)}] loss {loss.item():.4f} '
                    f'ce {l_ce.item():.4f} edlM {l_edl1.item():.4f} edlA {l_edl2.item():.4f} '
                    f'pear {ploss.item():+.4f} acc {agg["acc"]/n:.4f} lr {lr_now:.2e} '
                    f'| {(time.time()-t0)/60:.1f} min')

        tr = {k: v/max(n,1) for k, v in agg.items()}
        va_auc, va_acc, _, _ = evaluate(model, va_ld, len(val_videos))
        log(f'[EPOCH {epoch}] train loss {tr["loss"]:.4f} acc {tr["acc"]:.4f} pearson {tr["pear"]:+.4f} '
            f'| val video-AUC {100*va_auc:.2f} acc {100*va_acc:.2f} | {(time.time()-t0)/60:.1f} min')
        history.append({'epoch': epoch, **tr, 'val_auc': va_auc, 'val_acc': va_acc})

        if IS_MAIN:
            ck = {'epoch': epoch, 'state_dict': core.state_dict(), 'optimizer': optimizer.state_dict(),
                  'scaler': scaler.state_dict() if CFG['amp'] else None,
                  'best_auc': max(best_auc, va_auc), 'history': history, 'cfg': CFG}
            torch.save(ck, RUN_DIR/'last.pth')
            if va_auc > best_auc:
                torch.save({'epoch': epoch, 'state_dict': core.state_dict(),
                            'val_auc': va_auc, 'cfg': CFG, 'variant': VARIANT, 'flags': FL},
                           RUN_DIR/'best.pth')
                log(f'  -> best (val AUC {100*va_auc:.2f})')
            json.dump(history, open(RUN_DIR/'history.json', 'w'), indent=1)
        best_auc = max(best_auc, va_auc)
        if DDP: dist.barrier()

        if (time.time() - t_start)/3600 > CFG['max_train_hours']:
            log(f'ใกล้ชน time limit ที่ epoch {epoch} — หยุดไว้ก่อน รันใหม่เพื่อ resume')
            break

    # ประเมิน test ด้วย best checkpoint
    if (RUN_DIR/'best.pth').exists():
        core.load_state_dict(torch.load(RUN_DIR/'best.pth', map_location='cpu', weights_only=False)['state_dict'])
    te_auc, te_acc, s, l = evaluate(model, te_ld, len(test_videos))
    log(f'[FINAL] FF++ {A["compression"]} test video-AUC {100*te_auc:.2f} acc {100*te_acc:.2f}')
    if IS_MAIN:
        np.save(RUN_DIR/'test_scores.npy', np.stack([s, l]))
        json.dump({'test_auc': te_auc, 'test_acc': te_acc, 'best_val_auc': best_auc,
                   'variant': VARIANT, 'flags': FL},
                  open(RUN_DIR/'final.json', 'w'), indent=1)
    if DDP: dist.destroy_process_group()

main()
""", encoding="utf-8")
print("เขียนสคริปต์:", SCRIPT, f"({SCRIPT.stat().st_size/1024:.0f} KB)")

### 5.1 ข้อมูลที่ทุก variant ใช้ร่วมกัน (เหมือน 3b)


In [ ]:
base_payload = {
    "cfg": CFG,
    "repo_dir": str(REPO_DIR),
    "pretrain_path": str(pretrain_file),
    "compression": COMPRESSION,
    "train_items": train_frames,
    "val_videos":  [{"frames": r["frames"], "label": int(r["label"])} for _, r in index["val"].iterrows()],
    "test_videos": [{"frames": r["frames"], "label": int(r["label"])} for _, r in index["test"].iterrows()],
}
print(f"train frames {len(base_payload['train_items']):,} | val {len(base_payload['val_videos'])} | test {len(base_payload['test_videos'])} วิดีโอ")
eff_bs = CFG["batch_size"] * max(1, N_GPU) * CFG["grad_accum"]
print(f"effective batch size = {CFG['batch_size']} x {N_GPU} GPU x {CFG['grad_accum']} accum = {eff_bs}  (paper: 64)")
if eff_bs != 64:
    print("[WARN] ไม่เท่ากับ 64 ของ paper — ปรับ batch_size/grad_accum ใน CFG ให้ได้ 64")


## 6. เทรนทีละ variant

- variant ที่มี `final.json` อยู่แล้ว (จาก session ก่อนที่ Add Input ไว้) จะ**ข้าม**
- variant ที่ค้างกลางทาง (มี `last.pth`) จะ **resume**
- เทรนเสร็จแล้วจะลบ `last.pth` (2 GB) เหลือแค่ `best.pth` เพื่อไม่ให้ output เกิน 20 GB ของ Kaggle
- ถ้าเวลาไม่พอ variant ที่เหลือจะถูกข้าม → Save Version อีกรอบโดย Add Input เป็น output ของรอบนี้


In [ ]:
def restore_from_input(variant, run_dir):
    """คัดลอกผลของ variant นี้จาก session ก่อน (ถ้า Add Input ไว้)"""
    if not INPUT_DIR.exists():
        return "none"
    done = sorted(INPUT_DIR.rglob(f"ablation_seeds/{variant}/final.json"))
    if done:
        src = done[0].parent
        for f in ["best.pth", "final.json", "history.json", "train.log", "test_scores.npy", "config_meta.json"]:
            if (src / f).exists() and not (run_dir / f).exists():
                shutil.copyfile(src / f, run_dir / f)
        return "done"
    part = sorted(INPUT_DIR.rglob(f"ablation_seeds/{variant}/last.pth"))
    if part and not (run_dir / "last.pth").exists():
        src = part[0].parent
        for f in ["last.pth", "best.pth", "history.json", "train.log"]:
            if (src / f).exists():
                shutil.copyfile(src / f, run_dir / f)
        return "resume"
    return "none"

def run_variant(variant, seed):
    tag = f"{variant}_s{seed}"
    run_dir = RUN_ROOT / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    state = restore_from_input(tag, run_dir)
    if (run_dir / "final.json").exists():
        print(f"[{tag}] เสร็จแล้วจาก session ก่อน → ข้าม"); return 0
    left_h = NOTEBOOK_HOURS - (time.time() - T_START) / 3600
    need_h = 1.8 if VARIANT_FLAGS[variant]["dual"] else 1.0
    if left_h < need_h:
        print(f"[{variant}] เวลาเหลือ {left_h:.1f} ชม. ไม่พอ (ต้องการ ~{need_h} ชม.) → ข้าม รันต่อ session หน้า"); return None
    cfg = dict(CFG, seed=seed, max_train_hours=max(0.5, left_h - 0.4), variant=variant, flags=VARIANT_FLAGS[variant])
    payload = dict(base_payload, cfg=cfg, run_dir=str(run_dir), variant=variant, flags=VARIANT_FLAGS[variant])
    cfg_path = run_dir / "config.json"
    json.dump(payload, open(cfg_path, "w"))
    json.dump({"variant": variant, "seed": seed, "flags": VARIANT_FLAGS[variant], "pretrain_sha256": PRETRAIN_SHA,
               "repo_commit": REPO_COMMIT, "env": ENV}, open(run_dir / "config_meta.json", "w"), indent=1)
    print("=" * 70, f"\n[{tag}] seed={seed} flags={VARIANT_FLAGS[variant]} | state={state}", flush=True)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(N_GPU),
           "--master_port", "11023", str(SCRIPT), "--cfg", str(cfg_path)]
    env = {**os.environ, "OMP_NUM_THREADS": "2", "TOKENIZERS_PARALLELISM": "false",
           "PYTHONWARNINGS": "ignore", "NCCL_P2P_DISABLE": "1"}
    t0 = time.time()
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"[{variant}] exit code {rc} · {(time.time()-t0)/60:.1f} นาที")
    if rc == 0 and (run_dir / "final.json").exists():
        (run_dir / "last.pth").unlink(missing_ok=True)          # เก็บแค่ best.pth
        (run_dir / "config.json").unlink(missing_ok=True)       # config ใหญ่ (รายชื่อ frame) ไม่ต้องเก็บ
    elif rc != 0:
        print("[ERROR] ดู traceback ด้านบน · ถ้า CUDA out of memory: CFG['batch_size']=16, CFG['grad_accum']=2")
    return rc


In [ ]:
status = {}
for v, sd in JOBS:
    status[f"{v}_s{sd}"] = run_variant(v, sd)
print("\nสถานะ:", status)


## 7. ผลการเทรน

In [ ]:
rows = []
for v, sd in JOBS:
    rd = RUN_ROOT / f"{v}_s{sd}"; fj = rd / "final.json"
    if not fj.exists():
        rows.append({"variant": v, "seed": sd, "status": "ยังไม่เสร็จ"}); continue
    fin = json.load(open(fj))
    rows.append({"variant": v, "seed": sd, "status": "เสร็จ",
                 "best val AUC": round(100 * fin["best_val_auc"], 2),
                 "FF++ test AUC": round(100 * fin["test_auc"], 2), "FF++ test ACC": round(100 * fin["test_acc"], 2)})
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "seed_ffpp_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nseed 1234 (จาก 3b/3d): fb_ce val 97.31 / test 97.37 · cfb_full val 96.45 / test 96.72  (fused head สำหรับ 2 branch)")
print("ผล cross-dataset (main branch) และการทดสอบทางสถิติ → ระดับ 3f")
print(f"\nเวลารวม notebook: {(time.time()-T_START)/60:.1f} นาที")


## 8. ขั้นต่อไป

1. **Save Version → Save & Run All (Commit)** เพื่อเก็บ `ablation_seeds/<variant>_s<seed>/best.pth`
2. ถ้ายังไม่ครบทุก job → Add Input เป็น output ของรอบนี้ แล้ว Save Version อีกรอบ (job ที่เสร็จแล้วจะถูกข้าม)
3. ไปที่ **ระดับ 3f** — Add Input เป็น output ของ 3a, 3c, 3e และ 3d2 (ทุก account) แล้วรันการทดสอบทางสถิติ
